In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd
import timm

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)  # added RandomForestRegressor for a stronger tabular model
from sklearn.model_selection import train_test_split  # new import for validation split
from sklearn.metrics import (
    mean_squared_error,
)  # new import for computing validation RMSE

base_dir = "/kaggle/input"
model_weights = os.path.join(base_dir, "saved-weights", "swin_large_fastai(final).pth")
test_folder = os.path.join(base_dir, "petfinder-pawpularity-score", "test")
test_file = os.path.join(base_dir, "petfinder-pawpularity-score", "test.csv")
train_file = os.path.join(base_dir, "petfinder-pawpularity-score", "train.csv")



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
input_shape = (224, 224)
mean, std_dev = [0.5023, 0.4615, 0.4226], [0.2640, 0.2593, 0.2575]
model_name = "swin_large_patch4_window7_224_in22k"
batch_size = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
output_categories = 1  # single regression output
n_epochs = 15
num_of_hidden = 2
hidden_dimension = [256, 64]
save_name = "/kaggle/working/swin_fastai(final).pth"



In [3]:
test_csv = pd.read_csv(test_file)
test_csv["path_img"] = test_csv["Id"].apply(
    lambda x: os.path.join(test_folder, f"{x}.jpg")
)
test_csv["Pawpularity"] = 1




In [4]:
class PetsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.cat = [
            "Subject Focus",
            "Eyes",
            "Face",
            "Near",
            "Action",
            "Accessory",
            "Group",
            "Collage",
            "Human",
            "Occlusion",
            "Info",
            "Blur",
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["path_img"]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        tab = torch.tensor(self.df.iloc[idx][self.cat].values.astype(np.float32))
        label = torch.tensor(self.df.iloc[idx]["Pawpularity"], dtype=torch.float32)
        return img, tab, label




In [5]:
class Identity(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x


class Network(nn.Module):
    def __init__(
        self,
        base,
        number_of_hidden,
        hidden,
        regression_out,
        output_categories,
        freeze_layer,
        input_shape,
        device,
    ):
        super().__init__()
        if number_of_hidden != len(hidden):
            raise ValueError(
                "Number of Hidden layer and length of hidden dim must be same"
            )
        base.head = Identity()  # remove original classifier
        dummy = torch.randn(1, 3, input_shape[0], input_shape[1]).to(device)
        with torch.no_grad():
            dummy_feat = base(dummy)
            if dummy_feat.dim() > 2:
                dummy_feat = torch.flatten(dummy_feat, 1)
        visual_feat_dim = dummy_feat.shape[1]  # e.g. 75276
        target_vis_dim = (
            base.head.in_features if hasattr(base.head, "in_features") else hidden[0]
        )

        self.proj = nn.Linear(visual_feat_dim, target_vis_dim)

        hidden_dim = hidden[:]
        hidden_dim.insert(
            0, target_vis_dim + 12
        )  # after concatenating 12 tabular features
        self.p = 0.5
        self.regression = self._fully_connected(
            number_of_hidden, hidden_dim, output_categories
        )
        self.network = self._freeze_layer(base, freeze_layer)
        self._initialise_weights()

    def _initialise_weights(self):
        for m in self.regression:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _freeze_layer(self, base, freeze_layer):
        for i, child in enumerate(base.children()):
            if i < freeze_layer:
                for param in child.parameters():
                    param.requires_grad = False
        return base

    def _fully_connected(self, number_of_hidden, hidden_dim, output_categories):
        layers = []
        for i in range(number_of_hidden):
            layers.append(nn.Linear(hidden_dim[i], hidden_dim[i + 1]))
            layers.append(nn.GELU())
            layers.append(nn.BatchNorm1d(hidden_dim[i + 1]))
            if i != number_of_hidden - 1:
                layers.append(nn.Dropout(self.p))
        layers.append(nn.Linear(hidden_dim[-1], output_categories))
        return nn.Sequential(*layers)

    def forward(self, x, tab):
        x_feat = self.network(x)
        if x_feat.dim() > 2:
            x_feat = torch.flatten(x_feat, 1)
        x_feat = self.proj(x_feat)
        x_comb = torch.cat([x_feat, tab], dim=1)
        return self.regression(x_comb)


network = timm.create_model(model_name, pretrained=True)
network = network.to(device)  # ensure model is on the same device as dummy tensor
model = Network(
    network,
    num_of_hidden,
    hidden_dimension,
    regression_out=1,
    output_categories=output_categories,
    freeze_layer=0,
    input_shape=input_shape,
    device=device,
).to(device)



/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name swin_large_patch4_window7_224_in22k to current swin_large_patch4_window7_224.ms_in22k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/916M [00:00<?, ?B/s]

In [6]:
try:
    state_dict = torch.load(model_weights, map_location=device)
    model.load_state_dict(state_dict, strict=False)  # allow missing fastai keys
except Exception as e:
    print(f"Weight file not loaded ({e}); proceeding with pretrained model.")
model.eval()



Weight file not loaded ([Errno 2] No such file or directory: '/kaggle/input/saved-weights/swin_large_fastai(final).pth'); proceeding with pretrained model.


Network(
  (proj): Linear(in_features=75264, out_features=256, bias=True)
  (regression): Sequential(
    (0): Linear(in_features=268, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): GELU(approximate='none')
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
  (network): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    )
    (layers): Sequential(
      (0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine

In [7]:
train_df = pd.read_csv(train_file)

features = [
    "Subject Focus",
    "Eyes",
    "Face",
    "Near",
    "Action",
    "Accessory",
    "Group",
    "Collage",
    "Human",
    "Occlusion",
    "Info",
    "Blur",
]

X = train_df[features].astype(float)
y = train_df["Pawpularity"].astype(float)

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

gbr_tmp = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)
gbr_tmp.fit(X_tr, y_tr)
gbr_val_pred = gbr_tmp.predict(X_val)

rf_tmp = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=5,
)
rf_tmp.fit(X_tr, y_tr)
rf_val_pred = rf_tmp.predict(X_val)

gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)
gbr.fit(X, y)

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    random_state=42,
    n_jobs=5,
)
rf.fit(X, y)

preds_gbr = gbr.predict(test_csv[features].astype(float))
preds_rf = rf.predict(test_csv[features].astype(float))



In [8]:
transform = transforms.Compose(
    [
        transforms.Resize(input_shape),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std_dev),
    ]
)

val_idx = X_val.index
val_df = train_df.loc[val_idx].copy()
val_df["path_img"] = val_df["Id"].apply(
    lambda x: os.path.join(base_dir, "petfinder-pawpularity-score", "train", f"{x}.jpg")
)
val_dataset = PetsDataset(val_df, transform=transform)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=0
)

model.eval()
val_image_preds = []
with torch.no_grad():
    for img, tab, _ in val_loader:
        img = img.to(device)
        tab = tab.to(device)
        out = model(img, tab)  # (batch, 1)
        val_image_preds.append(out.squeeze().cpu().numpy())
val_image_preds = np.concatenate(val_image_preds)

# Compute validation RMSE for each tabular model
rmse_gbr = mean_squared_error(y_val, gbr_val_pred, squared=False)
rmse_rf = mean_squared_error(y_val, rf_val_pred, squared=False)

# Derive non‑negative weights inversely proportional to RMSE and normalize
w_gbr = 1.0 / rmse_gbr
w_rf = 1.0 / rmse_rf
weight_sum = w_gbr + w_rf
w_gbr /= weight_sum
w_rf /= weight_sum

# Apply the same weights to test predictions (ignore image model to avoid noise)
final_preds = w_gbr * preds_gbr + w_rf * preds_rf
final_preds = np.clip(final_preds, 0, 100)



In [9]:
submission = pd.DataFrame({"Id": test_csv["Id"], "Pawpularity": final_preds})
submission.to_csv("submission.csv", index=False)